In [3]:
from langchain_community.document_loaders import Docx2txtLoader

In [4]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_vertexai import ChatVertexAI

In [5]:
policy_prompt = ChatPromptTemplate.from_messages([
    ("system", 
     "you are an HR policy writer creating policies"
     "you will recieve an HR POLICY Template which is extracted from a DOCX file.\n"
     "Generate a New policy that follows same structure/headings style, but write ORIGINAL content.\n"
     "Rules:\n"
     "- Do not copy long phrases verbatiam from the template"
     "- Generate a concise and Well-structure policy"),
     ("user", 
      "TEMPLATE (reference): \n---\n{template_text}"
      "Generate policy:\n"
      "- company: {company_name}\n"
      "- company size: {company_size}\n"
      "- tone: {tone} \n"
      "- company category: {company_category}\n"
      "- country context: {country}\n"
      "Return only Markdown")

])

In [9]:
loader = Docx2txtLoader(r"/Users/munna/VScode/QT-AgenticAI/RAG/Enterprise_RAGS/HR_Policy_RAG/Policies/CompanyPolicies/Attendance Policy.docx")
docs = loader.load()
template_text = "\n\n".join(doc.page_content for doc in docs)


In [10]:

print(template_text)

Attendance Policy



1. OVERVIEW



Each employee at [Company Name] is responsible for punctual and consistent attendance. Employees should arrive on time, be prepared to work, and on schedule. Employees are also expected to stay at work for the whole of their shift. It is inconvenient to arrive late, leave early, or miss other scheduled hours, and it must be avoided.



This policy does not apply to FMLA-covered (FMLA - Family and Medical Leave Act) absences or leave taken as a reason

able accommodation under the Americans with Disabilities Act (ADA). They have their own policies that cover these exceptions.



2. OBJECTIVE

The goal of this policy is to lay out [Company Names] policies and processes for dealing with employee absences and tardiness in order to increase the company's efficiency and reduce unscheduled absences.



3. ATTENDANCE INFRACTIONS CALCULATION



Absent with calls - 1 Point 

Absent with no calls - 2 Points 

Tardy - ½ Point

Early Departure - ½ Point

Returnin

In [11]:
llm = ChatVertexAI(
    model_name="gemini-2.5-flash-lite",
    temperature=0.2,
    max_output_tokens=2048,
)

/var/folders/yx/b4_zr4nn6c701qx2kygmf_qc0000gn/T/ipykernel_31622/2543737531.py:1: DeprecationWarning: Use [`ChatGoogleGenerativeAI`][langchain_google_genai.ChatGoogleGenerativeAI] instead.
  llm = ChatVertexAI(
/var/folders/yx/b4_zr4nn6c701qx2kygmf_qc0000gn/T/ipykernel_31622/2543737531.py:1: LangChainDeprecationWarning: The class `ChatVertexAI` was deprecated in LangChain 3.2.0 and will be removed in 4.0.0. An updated version of the class exists in the `langchain-google-genai package and should be used instead. To use it run `pip install -U `langchain-google-genai` and import as `from `langchain_google_genai import ChatGoogleGenerativeAI``.
  llm = ChatVertexAI(
/Users/munna/VScode/QT-AgenticAI/RAG/Enterprise_RAGS/HR_Policy_RAG/.venv/lib/python3.13/site-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the fol

In [12]:
chain = policy_prompt | llm
response = chain.invoke(
    {
        "template_text": template_text,
        "company_name": "Acme Corp",
        "company_size": "10000 employees",
        "tone": "professional",
        "company_category": "Information Technology",
        "country": "India"
    })

/Users/munna/VScode/QT-AgenticAI/RAG/Enterprise_RAGS/HR_Policy_RAG/.venv/lib/python3.13/site-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


In [13]:
response.pretty_print()

================================== Ai Message ==================================

```markdown
# Employee Punctuality and Absence Policy

## 1. Introduction

Acme Corp values the dedication and commitment of its employees. Punctual and consistent attendance is crucial for maintaining operational efficiency, team collaboration, and overall productivity within our Information Technology environment. This policy outlines the expectations and procedures regarding employee attendance, tardiness, and absences.

This policy does not supersede or replace provisions related to legally protected leaves such as those under the Maternity Benefit Act, Employees' State Insurance Act, or other applicable Indian labor laws, nor does it cover absences approved as reasonable accommodations under relevant disability legislation.

## 2. Purpose

The objective of this policy is to establish clear guidelines and a consistent framework for managing employee attendance and absence. This aims to minimize disrup

In [14]:

import os

def combine_dir_with_markdown(dir_path, docx_path):
    # Extract filename from docx path
    filename = os.path.basename(docx_path)
    
    # Remove extension and convert to markdown name
    name_without_ext = os.path.splitext(filename)[0]
    markdown_name = name_without_ext.replace(" ", "_") + ".md"
    
    # Combine directory path with markdown filename
    return os.path.join(dir_path, markdown_name)

In [16]:
from langchain_community.document_loaders import DirectoryLoader
chain = policy_prompt | llm

directory_loader = DirectoryLoader(
    "/Users/munna/VScode/QT-AgenticAI/RAG/Enterprise_RAGS/HR_Policy_RAG/Policies/CompanyPolicies",
    glob="*.docx",
    loader_cls=Docx2txtLoader)

docs = directory_loader.load()
for doc in docs:
    template_text = "\n\n".join(doc.page_content for doc in docs)
    path = combine_dir_with_markdown(
        "/Users/munna/VScode/QT-AgenticAI/RAG/Enterprise_RAGS/HR_Policy_RAG/GeneratedPolicies", doc.metadata['source'])
    response = chain.invoke(
        {
            "template_text": template_text,
            "company_name": "Acme Corp",
            "company_size": "10000 employees",
            "tone": "professional",
            "company_category": "Information Technology",
            "country": "India"
        })
    with open(path, 'w') as f:
        f.write(response.content)